# 🏒🌍 European and college hockey with `sportsdataverse-py`

North America's minor and junior leagues have their own notebook (`11_junior_hockey_intro`). This one
crosses the Atlantic and drops into the NCAA:

| League | Module | Provider |
|---|---|---|
| SHL (Sweden) | `sportsdataverse.hockey.shl` | shl.se |
| Liiga (Finland) | `sportsdataverse.hockey.liiga` | liiga.fi |
| DEL (Germany) | `sportsdataverse.hockey.del_` | DEL (eSports feed) |
| KHL | `sportsdataverse.hockey.khl` | khl.ru |
| NCAA men's / women's hockey | `sportsdataverse.hockey.mch` / `.wch` | ESPN |

DEL's package is `del_`, because `del` is a Python keyword. The European runtimes take the season as
its END year (2026 is 2025-26) and space requests at least a second apart, so be patient. ⏳

## 🧰 The toolbox

| Function | What it gives you |
|---|---|
| [`shl_schedule`](../shl/reference/shl.md#shl_schedule) | Schedule and results of an SHL season |
| [`shl_pbp`](../shl/reference/shl.md#shl_pbp) | SHL play-by-play with shot locations (2023/24 on) |
| [`liiga_standings`](../liiga/reference/liiga.md#liiga_standings) | Liiga standings |
| [`liiga_games`](../liiga/reference/liiga.md#liiga_games) | Every Liiga game of a season, with xG |
| [`liiga_shotmap`](../liiga/reference/liiga.md#liiga_shotmap) | One Liiga game's shots |
| [`del_schedule`](../del/reference/del_esports.md#del_schedule) | DEL schedule and results |
| [`del_standings`](../del/reference/del_esports.md#del_standings) | DEL standings |
| [`del_shots`](../del/reference/del_esports.md#del_shots) | One DEL game's shots with rink coordinates |
| [`khl_standings`](../khl/reference/khl_rest.md#khl_standings) | KHL regular-season standings |
| [`espn_mch_standings`](../mch/reference/site.md#espn_mch_standings) | NCAA men's hockey standings |
| [`espn_wch_scoreboard`](../wch/reference/site.md#espn_wch_scoreboard) | NCAA women's hockey scoreboard |

## 🔌 Setup

```sh
pip install "sportsdataverse[sdvplot]"
```

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from sportsdataverse.hockey import del_, khl, liiga, mch, shl, wch

pl.Config.set_tbl_rows(10)
SEASON = 2026  # END year: the completed 2025-26 season

Provider feeds are seasonal and occasionally rate-limited, so a tiny `safe()` helper runs each call
defensively: you get the frame when the feed is up, and a one-line skip when it isn't, naming whether
the answer was *nothing there* (`NoDataError`) or *the fetch failed* (`AssetFetchError`). 🛟

In [ ]:
from sportsdataverse.errors import AssetFetchError, NoDataError

def safe(label, thunk):
    try:
        out = thunk()
        print(f"✅ {label}")
        return out
    except (NoDataError, AssetFetchError) as e:
        print(f"\u23ed\ufe0f  {label}: {type(e).__name__}: {e}")
        return None

## 🇸🇪 SHL

[`shl_schedule`](../shl/reference/shl.md#shl_schedule) returns every game of a season with its
`game_uuid`; [`shl_pbp`](../shl/reference/shl.md#shl_pbp) takes that uuid.

In [ ]:
shl_sched = safe(f"SHL schedule {SEASON}", lambda: shl.shl_schedule(SEASON))
cols = ["start_date_time", "home_team_code", "home_team_score", "away_team_code", "away_team_score", "state"]
shl_sched.select([c for c in cols if c in shl_sched.columns]).tail(5) if shl_sched is not None else None

In [ ]:
done = shl_sched.filter(pl.col("state") == "post-game") if shl_sched is not None else None
guid = done["game_uuid"][-1] if done is not None and done.height else None
shl_pbp = safe("SHL pbp", lambda: shl.shl_pbp(guid)) if guid else None
(shl_pbp["type"].value_counts(sort=True) if shl_pbp is not None and shl_pbp.height
 else "no SHL play-by-play")

## 🇫🇮 Liiga

[`liiga_standings`](../liiga/reference/liiga.md#liiga_standings) defaults to the current season.
[`liiga_games`](../liiga/reference/liiga.md#liiga_games) carries both teams' goals per game, and
[`liiga_shotmap`](../liiga/reference/liiga.md#liiga_shotmap) one game's shots.

In [ ]:
lst = safe("Liiga standings", liiga.liiga_standings)
cols = ["tournament", "ranking", "team_name", "games", "wins", "losses", "points", "goals", "goals_against"]
(lst.filter(pl.col("tournament") == lst["tournament"][0])
    .select([c for c in cols if c in lst.columns]).sort("ranking").head(8)
 if lst is not None and lst.height else "standings unavailable")

In [ ]:
lg = safe("Liiga games", liiga.liiga_games)
fin = lg.filter(pl.col("ended") == True) if lg is not None else None
if fin is not None and fin.height:
    g = fin.row(-1, named=True)
    sm = safe(f"Liiga shotmap {g['game_id']}", lambda: liiga.liiga_shotmap(g["season"], g["game_id"]))
    print(g["home_name"], g["home_goals"], "-", g["away_goals"], g["away_name"])
    out = sm.group_by("event_type").len() if sm is not None and sm.height else "no shotmap"
else:
    out = "no finished Liiga games"
out

## 🇩🇪 DEL

[`del_standings`](../del/reference/del_esports.md#del_standings) and
[`del_schedule`](../del/reference/del_esports.md#del_schedule) take the END year;
[`del_shots`](../del/reference/del_esports.md#del_shots) a `match_id`.

In [ ]:
dst = safe(f"DEL standings {SEASON}", lambda: del_.del_standings(SEASON))
cols = ["position", "name", "games", "wins", "wins_ot", "losses_ot", "losses", "points"]
dst.select([c for c in cols if c in dst.columns]).sort("position").head(8) if dst is not None else None

### A DEL shot map 🎯

`del_shots` ships `coordinate_x` / `coordinate_y` in the provider's own frame (both axes run about
-100 to 100), plus `team_side` and `is_goal`. sdvplot has no mapping from this frame to a regulation
IIHF rink yet, so the chart below plots the raw frame rather than drawing a rink under it.

In [ ]:
dsched = safe(f"DEL schedule {SEASON}", lambda: del_.del_schedule(SEASON))
mid = int(dsched.filter(pl.col("final_score_home").is_not_null())["match_id"][-1]) if dsched is not None else None
dshots = safe(f"DEL shots {mid}", lambda: del_.del_shots(mid)) if mid else None
if dshots is not None and dshots.height:
    fig, ax = plt.subplots(figsize=(8, 5))
    for side, color in (("home", "#1f77b4"), ("visitor", "#ff7f0e")):
        s = dshots.filter(pl.col("team_side") == side)
        name = s["home_name" if side == "home" else "visitor_name"][0] if s.height else side
        ax.scatter(s["coordinate_x"], s["coordinate_y"], s=25, alpha=0.6, c=color, label=name)
    goals = dshots.filter(pl.col("is_goal") == True)
    ax.scatter(goals["coordinate_x"], goals["coordinate_y"], s=160, facecolors="none",
               edgecolors="black", linewidths=1.5, label="goal")
    ax.set_xlim(-105, 105)
    ax.set_ylim(-105, 105)
    ax.set_aspect("equal")
    ax.legend(loc="upper center", ncol=3, fontsize=8)
    ax.set_title(f"DEL match {mid}: {dshots.height} shots (provider frame)")
    plt.show()
else:
    print("no DEL shots for this match")

## 🇷🇺 KHL

[`khl_standings`](../khl/reference/khl_rest.md#khl_standings) returns one row per team per table
(conference and division tables are stacked, tagged by `table`).

In [ ]:
kst = safe("KHL standings", khl.khl_standings)
cols = ["table", "rank", "name", "gp", "w", "l", "pts", "goals_for", "goals_against"]
(kst.filter(pl.col("table") == kst["table"][0]).select([c for c in cols if c in kst.columns]).head(8)
 if kst is not None and kst.height else "standings unavailable")

## 🎓 College hockey (ESPN)

The NCAA men's (`mch`) and women's (`wch`) leagues ride the cross-league ESPN wrappers. Team colors
come from sdvplot's `team_colors`, keyed by ESPN team id under the `ncaa_mhockey` league key.

In [ ]:
import sdvplot

cst = safe("NCAA men's hockey standings", mch.espn_mch_standings)
if cst is not None and cst.height:
    top = cst.sort("wins", "win_percent", descending=True).head(15).sort("wins", "win_percent")
    colors = sdvplot.team_colors("ncaa_mhockey", [str(i) for i in top["team_id"]])
    colors = [c or "#999999" for c in colors]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(top["team_display_name"], top["wins"], color=colors)
    ax.set_xlabel("wins")
    ax.set_title("NCAA men's hockey: the 15 teams with the most wins so far")
    plt.tight_layout()
    plt.show()
else:
    print("standings unavailable")

In [ ]:
wsb = safe("NCAA women's hockey scoreboard", wch.espn_wch_scoreboard)
cols = ["date", "short_name", "status_type_detail", "home_score", "away_score"]
wsb.select([c for c in cols if c in wsb.columns]) if wsb is not None and wsb.height else "no games today"

## 🎉 Where to next

- More European homes under `sportsdataverse.hockey`: `icehl`, `eihl`, `metal`, `elh`, `ha`, `nl`,
  `magnus`, `vhl` and the Swiss `sihf`; each has a reference page in the sidebar.
- `khl_webcaster_*` is a second KHL source with player and team tables per stage.
- North American minor and junior leagues: `11_junior_hockey_intro`; the PWHL: `10_pwhl_intro`; the
  NHL: `07_nhl_intro`.
- Rinks and team colors: [sdvplot](https://sdvplot.sportsdataverse.org/) and
  [sportypy](https://sportypy.sportsdataverse.org/).